# Credit Scoring Model - Exploratory Data Analysis

This notebook conducts Exploratory Data Analysis (EDA) on the Credit Risk dataset to analyze borrower defaults and builds a baseline Logistic Regression classifier.

### 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

sns.set_theme(style="whitegrid")
%matplotlib inline

### 2. Load the Dataset

In [ ]:
# Make sure you have run python main.py once or the scratch data generator is executed to create the dataset
data_path = '../data/sample_credit_data.csv'
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
df.head()

### 3. Basic Dataset Info & Null Check

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
df.isna().sum()

### 4. Target Variable Distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='loan_status', hue='loan_status', data=df, palette='Set2', legend=False)
plt.title('Distribution of Loan Status (0 = Non-Default, 1 = Default)')
plt.xlabel('Loan Status')
plt.ylabel('Count')
plt.show()

default_rate = df['loan_status'].mean()
print(f"Baseline Default Rate: {default_rate:.2%}")

### 5. Numerical Feature Explorations

In [ ]:
# Correlation of numerical features
plt.figure(figsize=(10, 8))
# Only correlate numerical cols
num_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

In [ ]:
# Box plots of Key Features vs Loan Status
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(ax=axes[0], x='loan_status', y='person_income', hue='loan_status', data=df, palette='Set1', legend=False)
axes[0].set_title('Income vs Loan Status')
axes[0].set_yscale('log') # Income distributions are highly skewed

sns.boxplot(ax=axes[1], x='loan_status', y='loan_int_rate', hue='loan_status', data=df, palette='Set1', legend=False)
axes[1].set_title('Interest Rate vs Loan Status')

sns.boxplot(ax=axes[2], x='loan_status', y='loan_percent_income', hue='loan_status', data=df, palette='Set1', legend=False)
axes[2].set_title('Loan to Income Ratio vs Loan Status')

plt.tight_layout()
plt.show()

### 6. Categorical Feature Analysis

In [ ]:
# Analyze loan grade default rates
plt.figure(figsize=(8, 4))
grade_default = df.groupby('loan_grade')['loan_status'].mean().reset_index()
sns.barplot(x='loan_grade', y='loan_status', hue='loan_grade', data=grade_default.sort_values(by='loan_grade'), palette='Reds', legend=False)
plt.title('Default Rate by Loan Grade')
plt.ylabel('Default Rate')
plt.xlabel('Loan Grade')
plt.show()

In [ ]:
# Analyze Home Ownership default rates
plt.figure(figsize=(8, 4))
home_default = df.groupby('person_home_ownership')['loan_status'].mean().reset_index()
sns.barplot(x='person_home_ownership', y='loan_status', hue='person_home_ownership', data=home_default, palette='Blues', legend=False)
plt.title('Default Rate by Home Ownership Type')
plt.ylabel('Default Rate')
plt.xlabel('Home Ownership')
plt.show()

### 7. Baseline Modeling

In [ ]:
# Handing missing data in a basic way
df_model = df.copy()
df_model['person_emp_length'] = df_model['person_emp_length'].fillna(df_model['person_emp_length'].median())
df_model['loan_int_rate'] = df_model['loan_int_rate'].fillna(df_model['loan_int_rate'].median())

# Split features and target
X = df_model.drop(columns=['loan_status'])
y = df_model['loan_status']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

categorical_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
numerical_cols = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Train Baseline Logistic Regression
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_proc, y_train)

# Predict
y_pred = baseline_model.predict(X_test_proc)
y_prob = baseline_model.predict_proba(X_test_proc)[:, 1]

# Evaluate
print("Baseline Logistic Regression classification report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print(f"Gini Coefficient: {(2 * roc_auc_score(y_test, y_prob) - 1):.4f}")

### 8. ROC Curve Plotting

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Baseline ROC (AUC = {roc_auc_score(y_test, y_prob):.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Baseline ROC Curve')
plt.legend(loc="lower right")
plt.show()